有db版本 1. Installation  & 2. Library Import and Configuration

In [ ]:
%pip install duckdb leafmap lonboard
import duckdb
import leafmap
import pandas as pd

3. Sample Data

In [ ]:
url = "https://storage.googleapis.com/qm2/CASA0025/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

4.Connecting to DuckDB

In [ ]:
con = duckdb.connect("nyc_data.db")

con.install_extension("spatial")
con.load_extension("spatial")

In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
con.sql("""
describe nyc_neighborhoods
""")

In [ ]:
con.sql("SELECT * from nyc_subway_stations LIMIT 5")

1. Installation nonono db，csv版本  2.Library Import

In [ ]:
%pip install duckdb leafmap
import duckdb
import leafmap
import pandas as pd

3.Installing Extensions

In [ ]:
con = duckdb.connect()

con.install_extension("httpfs")
con.load_extension("httpfs")

con.install_extension("spatial")
con.load_extension("spatial")

4.Downloading Sample Data

In [ ]:
url = "https://storage.googleapis.com/qm2/CASA0025/cities.zip"
leafmap.download_file(url, unzip=True, overwrite=True)

5. read csv

In [ ]:
con.sql("SELECT * FROM 'cities.csv'")

6. create table

In [ ]:
con.sql("CREATE TABLE cities AS SELECT * FROM ST_Read('cities.geojson')")

In [ ]:
con.sql("""
CREATE TABLE IF NOT EXISTS nyc_neighborhoods AS
SELECT *
FROM read_csv_auto('postgis-workshop/data/nyc_neighborhoods.csv');
""")

In [ ]:
con.sql("""
CREATE TABLE countries AS SELECT * FROM 'https://storage.googleapis.com/qm2/CASA0025/countries.csv';
""")

In [ ]:
con.sql("""
CREATE TABLE IF NOT EXISTS nyc_neighborhoods AS
SELECT *
FROM st_read('postgis-workshop/data/nyc_neighborhoods.shp');

CREATE TABLE IF NOT EXISTS nyc_subway_stations AS
SELECT *
FROM st_read('postgis-workshop/data/nyc_subway_stations.shp');
""")


In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
con.sql("SELECT * FROM cities LIMIT 5;")

In [ ]:
con.sql("DESCRIBE cities;")

In [ ]:
con.sql("""


""")

空间索引

In [ ]:
con.sql("""
CREATE INDEX gaza_buildings_geom_idx
ON gaza_buildings
USING RTREE (geom);
""")

1. 基础聚合函数 (Aggregates)
sum(expression),计算一组记录的总和。
count(expression),计算一组记录的数量（大小）。

In [ ]:
2. 几何属性查询 (Geometry Properties)
ST_GeometryType(geom)	返回几何体类型（如 Point, LineString, Polygon）。
ST_NDims(geom)	返回几何体的维度。
ST_SRID(geom)	返回空间参考识别号（坐标系 ID）。
ST_X(point) / ST_Y(point)	返回点的 X（经度） 或 Y（纬度） 坐标。
ST_NPoints(geom)	返回几何体包含的坐标点数量。

In [ ]:
3. 测量函数 (Measurement Functions)
ST_Length(linestring)	计算线的长度。
ST_Area(polygon)	计算多边形的面积。
ST_Perimeter(geom)	计算所有环的总周长。
ST_Distance(geom A, geom B)	计算 A 和 B 之间的最短距离。

In [ ]:
4. 几何体组成与访问 (Geometry Accessors)
ST_StartPoint(geom)	返回线段的起点。
ST_EndPoint(geom)	返回线段的终点。
ST_NRings(polygon)	返回多边形环的数量（1个外环 + N个内环/洞）。
ST_ExteriorRing(poly)	返回多边形的外环（结果为线）。
ST_InteriorRingN(poly, n)	返回指定的第 n 个内环。
ST_NumGeometries(coll)	返回几何对象集合中子部分的数量。
ST_GeometryN(coll, n)	返回集合中指定的第 n 个子部分。

In [ ]:
5. 空间关系判断 (Spatial Relationships)
ST_Intersects(A, B)	最常用。只要 A 和 B 有任何接触或重叠，返回 True。
ST_Disjoint(A, B)	A 和 B 完全不相交（与 Intersects 相反）。
ST_Contains(A, B)	A 是否包含 B。
ST_Within(A, B)	A 是否在 B 内部。
ST_DWithin(A, B, r)	A 和 B 的距离是否在半径 r 范围内（比 Distance 后过滤更高效）。
ST_Touches(A, B)	A 和 B 仅在边界接触，内部不重叠。
ST_Crosses(A, B)	A 和 B 交叉（通常指线与线，或线与面）。
ST_Overlaps(A, B)	A 和 B 空间重叠（相同维度且不完全包含）。
ST_Equals(A, B)	A 和 B 几何上完全相同。

In [ ]:
6. 格式转换 (Output & Input)
ST_GeomFromText(text),将 WKT 文本（如 'POINT(0 0)'）转为几何对象。
ST_AsText(geom),将几何对象转为 WKT（易读文本）。
ST_AsGeoJSON(geom),转为 JSON 格式（网页前端常用）。
ST_AsBinary(geom),转为 WKB（二进制，适合程序传输）。
ST_AsKML / ST_AsGML / ST_AsSVG,分别转为 KML (Google Earth)、GML 或 SVG 矢量图格式。

In [ ]:
con.sql("""
select BORONAME, NAME from nyc_neighborhoods
where ST_Intersects(geom, ST_GeomFromText(
  'SRID=26918;LINESTRING(586782 4504202, 586864 4504216)'
))
""")

In [ ]:
con.sql("""
SELECT name
FROM nyc_streets
WHERE ST_DWithin(
        geom,
        ST_GeomFromText('POINT(583571 4506714)'),
        10
      );
""")

In [ ]:
con.sql("""
SELECT DISTINCT n.name, n.boroname
FROM nyc_subway_stations AS s
JOIN nyc_neighborhoods AS n
ON ST_Contains(n.geom, s.geom)
WHERE strpos(s.routes,'6') > 0;
""")

In [ ]:
SELECT Sum(popn_total)
FROM nyc_neighborhoods AS n
JOIN nyc_census_blocks AS c
ON ST_Intersects(n.geom, c.geom)
WHERE n.name = 'Battery Park';

In [ ]:
ST_DWithin_Spheroid → 距离单位 = 米